# Create the sumstats file and snplist in the correct format for ldsc

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Adjust this path to where config.py is located
code_dir = Path("/cluster/work/boeva/lrabuzin/deepcast/src")  # or something relative like ../code
sys.path.append(str(code_dir))

### 1. Get sumstats with `Z`, `SNP`, and `N` columns:

In [ ]:
# snplist:
import json
import pandas as pd
from pathlib import Path

from config import ALT_COL, BP_COL, CHR_COL, REF_COL, REF_DICT_PATH, REF_DIR_1KG, CHROMOSOMES, SNP_COL

out_folder=Path("/cluster/work/boeva/lrabuzin/deepcast/data/1kg_reference_genome")

with open(REF_DICT_PATH, 'r') as f:
    reference_filename_by_chr = json.load(f)
    reference_filename_by_chr = {int(k): v for k, v in reference_filename_by_chr.items()}

for i in CHROMOSOMES:
    reference_filename = reference_filename_by_chr[i]
    dir_1kg = Path(REF_DIR_1KG)
    reference_file_path = dir_1kg / reference_filename
    out_name = '_'.join(reference_filename.split('_')[0:2])+'_reference.csv' # drop '_combined' and replace it with '_reference'
    try:
        chunk = pd.read_csv(reference_file_path, usecols=[CHR_COL, BP_COL, REF_COL, ALT_COL, SNP_COL])
        chunk.to_csv(out_folder / out_name)
    except Exception as e:
        print(f"Error reading {reference_file_path}: {e}")
    

In [1]:
# snplist:
import json
import pandas as pd
from pathlib import Path

from config import ALT_COL, BP_COL, CHR_COL, REF_COL, REF_DICT_PATH, REF_DIR_1KG, CHROMOSOMES, SNP_COL

out_folder=Path("/cluster/work/boeva/lrabuzin/deepcast/data/1kg_reference_genome")

with open(REF_DICT_PATH, 'r') as f:
    reference_filename_by_chr = json.load(f)
    reference_filename_by_chr = {int(k): v for k, v in reference_filename_by_chr.items()}

dataframes=[]
for i in CHROMOSOMES:
    reference_filename = reference_filename_by_chr[i]
    dir_1kg = Path(REF_DIR_1KG)
    reference_file_path = dir_1kg / reference_filename
    out_name = '_'.join(reference_filename.split('_')[0:2])+'_reference.csv' # drop '_combined' and replace it with '_reference'
    try:
        chunk = pd.read_csv(reference_file_path, usecols=[CHR_COL, BP_COL, REF_COL, ALT_COL, SNP_COL])
        dataframes.append(chunk)
    except Exception as e:
        print(f"Error reading {reference_file_path}: {e}")    

KeyboardInterrupt: 

In [9]:
import pandas as pd
from glob import glob

paths = sorted(glob("/cluster/work/boeva/lrabuzin/deepcast/data/1kg_reference_genome/*.csv"))
dfs = [pd.read_csv(p, sep=",", usecols=['snp', 'ref', 'alt']) for p in paths]

# Make sure columns are named right
for i, d in enumerate(dfs):
    d.rename(columns={"snp":"SNP","ref":"A1","alt":"A2"}, inplace=True)

# Combine and sanitize
all_snps = (pd.concat(dfs, ignore_index=True)
              # .dropna(subset=["SNP","A1","A2"])
              # .assign(A1=lambda x: x["A1"].str.upper(),
              #         A2=lambda x: x["A2"].str.upper(),
              #         SNP=lambda x: x["SNP"].astype(str))
              )

# Write *without header*, tab-delimited (what ldsc expects)
# all_snps[["SNP","A1","A2"]].to_csv("my_merge_alleles.txt", sep="\t", index=False, header=False)

# (Optional) gzip to save space; ldsc handles .gz
all_snps[["SNP","A1","A2"]].to_csv("snps_1kg.txt.gz", sep="\t", index=False, header=True, compression="gzip")


In [7]:
reference_filename="1000G.MAF_threshold=0.005.1_combined.csv"
out_name = '_'.join(reference_filename.split('_')[0:2])+'_reference.csv'
out_name

'1000G.MAF_threshold=0.005.1_reference.csv'

### 2. Merge sumstats with reference to get SNP id

In [2]:
from input_output import read_in_sumstats, retrieve_sumstats_filename, retrieve_n_cases

phen=4559

sumstats = read_in_sumstats(filename=retrieve_sumstats_filename(phen), column_names={}, read_in_beta=True)

14:48:16 - input_output - INFO - Reading in sumstats
14:48:16 - input_output - DEBUG - sumstats path /cluster/work/boeva/lrabuzin/deepcast/data/ukbb_phens/icd10_sumstats_/icd10-C02-both_sexes.tsv.bgz
14:48:54 - input_output - DEBUG - Found 5125720 of 28987534 N/A rows in column neglog10_pval_EUR.
14:48:54 - input_output - INFO - Dropping N/A rows and resetting the index


In [3]:
from ldsc_files.merge_sumstats import merge_sumstats_reference

merged_sumstats = merge_sumstats_reference(sumstats)

14:49:20 - ldsc_files.merge_sumstats - INFO - Merge summary stats with enformer tracks by chromosome:
14:49:38 - ldsc_files.merge_sumstats - INFO - Processed file 1000G.MAF_threshold=0.005.1_combined.csv
14:49:42 - ldsc_files.merge_sumstats - INFO - Processed file 1000G.MAF_threshold=0.005.3_combined.csv
14:49:42 - ldsc_files.merge_sumstats - INFO - Processed file 1000G.MAF_threshold=0.005.2_combined.csv
14:49:44 - ldsc_files.merge_sumstats - INFO - Processed file 1000G.MAF_threshold=0.005.4_combined.csv
14:49:44 - ldsc_files.merge_sumstats - INFO - Processed file 1000G.MAF_threshold=0.005.5_combined.csv
14:49:46 - ldsc_files.merge_sumstats - INFO - Processed file 1000G.MAF_threshold=0.005.6_combined.csv
14:49:48 - ldsc_files.merge_sumstats - INFO - Processed file 1000G.MAF_threshold=0.005.7_combined.csv
14:49:49 - ldsc_files.merge_sumstats - INFO - Processed file 1000G.MAF_threshold=0.005.9_combined.csv
14:49:49 - ldsc_files.merge_sumstats - INFO - Processed file 1000G.MAF_threshold=0

In [4]:
merged_sumstats['N'] = retrieve_n_cases(phen)

In [5]:
merged_sumstats.head(10)

,chr,pos,ref,alt,beta_EUR,neglog10_pval_EUR,snp,N
0,1,11063,T,G,-0.6591,0.01185,rs561109771,259.0
1,1,13259,G,A,-1.1430,0.16400,NaN,259.0
2,1,17641,G,A,-1.1290,0.28700,NaN,259.0
3,1,57222,T,C,2.0190,0.59470,NaN,259.0
4,1,58396,T,C,-1.0020,0.13850,NaN,259.0
5,1,63668,G,A,-1.2750,0.04988,NaN,259.0
6,1,69569,T,C,-1.0970,0.13050,NaN,259.0
7,1,79192,T,G,-1.0740,0.07864,NaN,259.0
8,1,91588,G,A,-1.0140,0.11060,NaN,259.0
9,1,533573,G,A,2.0270,0.61060,rs575442534,259.0


In [6]:
snps = merged_sumstats.dropna(subset=['snp']).reset_index(drop=True)

In [7]:
snps.head(10)

,chr,pos,ref,alt,beta_EUR,neglog10_pval_EUR,snp,N
0,1,11063,T,G,-0.65910,0.011850,rs561109771,259.0
1,1,533573,G,A,2.02700,0.610600,rs575442534,259.0
2,1,541944,T,C,-1.23300,0.219500,rs568792105,259.0
3,1,672798,G,A,-1.61600,0.056940,rs538740799,259.0
4,1,691545,A,T,-3.53700,0.134000,rs548729314,259.0
5,1,692794,CA,C,0.01005,0.023050,rs530212009,259.0
6,1,693731,A,G,0.05681,0.156000,rs12238997,259.0
7,1,706425,C,T,-1.33500,0.103000,rs529439608,259.0
8,1,706992,C,T,-0.33380,0.008095,rs533042087,259.0
9,1,707014,A,C,-0.18140,0.004337,rs141817527,259.0


In [8]:
len(snps)

15661089

### 3. Compute Z-scores (is this the chi-square statistics or the z-scores for the normal distribution??) from p-value

In [12]:
snps.columns

Index(['chr', 'pos', 'ref', 'alt', 'beta_EUR', 'neglog10_pval_EUR', 'snp',
       'N'],
      dtype='object')

In [ ]:
import numpy as np
from scipy.stats import norm
from scipy.stats import chi2
from utils import pval_from_neglog10

# hopefully fine but I'm not sure:
snps['P'] = pval_from_neglog10(snps)
snps["my_Z"] = np.sign(snps['beta_EUR']) * norm.isf(snps["P"] / 2)
snps['Z'] = np.sqrt(chi2.isf(snps["P"], 1))

In [19]:
snps.head()

,chr,pos,ref,alt,beta_EUR,neglog10_pval_EUR,snp,N,P,Z
0,1,11063,T,G,-0.6591,0.01185,rs561109771,259.0,0.973083,-0.033742
1,1,533573,G,A,2.0270,0.61060,rs575442534,259.0,0.245132,1.162255
2,1,541944,T,C,-1.2330,0.21950,rs568792105,259.0,0.603254,-0.519727
3,1,672798,G,A,-1.6160,0.05694,rs538740799,259.0,0.877122,-0.154619
4,1,691545,A,T,-3.5370,0.13400,rs548729314,259.0,0.734514,-0.339127


In [28]:
snps[snps['snp']=='rs6657544']

,chr,pos,ref,alt,beta_EUR,neglog10_pval_EUR,snp,N,P,Z
3225,1,1186665,G,A,-0.0274,0.07724,rs6657544,259.0,0.837067,-0.205647
3226,1,1186665,G,T,0.4484,0.49740,rs6657544,259.0,0.318127,0.998315


### 4. Save sumstats

In [29]:
sumstats_formatted = snps.rename(columns={'snp':'SNP', 'ref':'A1', 'alt':'A2'})[['Z', 'SNP', 'N', 'A1', 'A2']]

In [30]:
sumstats_formatted.head()

,Z,SNP,N,A1,A2
0,-0.033742,rs561109771,259.0,T,G
1,1.162255,rs575442534,259.0,G,A
2,-0.519727,rs568792105,259.0,T,C
3,-0.154619,rs538740799,259.0,G,A
4,-0.339127,rs548729314,259.0,A,T


In [25]:
sumstats_formatted[sumstats_formatted['SNP'].duplicated()]

,Z,SNP,N
3226,0.998315,rs6657544,259.0
3510,1.713089,rs6666790,259.0
4306,2.045233,rs139912116,259.0
6822,0.019986,rs28657980,259.0
9393,0.827684,rs11589451,259.0
...,...,...,...
15657815,-0.300909,rs9626401,259.0
15658264,1.018906,rs9614750,259.0
15659059,1.331640,rs713681,259.0
15659692,-0.310788,rs10453440,259.0


In [31]:
sumstats_formatted[sumstats_formatted['SNP']=='rs6657544']

,Z,SNP,N,A1,A2
3225,-0.205647,rs6657544,259.0,G,A
3226,0.998315,rs6657544,259.0,G,T


### 5. Check output

In [7]:
import gzip
import pandas as pd

path = '/cluster/work/boeva/lrabuzin/deepcast/benchmarking_study/reformatted_sumstats/run_7391322/icd10-C02-both_sexes.tsv.bgz'

file = gzip.open(path, "rt")
formatted_sumstats = pd.concat(pd.read_csv(file, sep="\t", chunksize=100000), ignore_index=True) # usecols=selected_columns, 
formatted_sumstats

,Z,SNP,N,P,A1,A2
0,-0.033742,rs561109771,259.0,0.973083,T,G
1,1.162255,rs575442534,259.0,0.245132,G,A
2,-0.519727,rs568792105,259.0,0.603254,T,C
3,-0.154619,rs538740799,259.0,0.877122,G,A
4,-0.339127,rs548729314,259.0,0.734514,A,T
...,...,...,...,...,...,...
15661084,-1.084599,rs185068034,259.0,0.278099,G,T
15661085,1.052691,rs62225896,259.0,0.292483,G,A
15661086,-0.085588,rs114414268,259.0,0.931794,G,C
15661087,0.486091,rs12160757,259.0,0.626902,T,C


In [14]:
import gzip
import pandas as pd

path = '/cluster/work/boeva/lrabuzin/deepcast/benchmarking_study/snps_1kg.txt.gz'

file = gzip.open(path, "rt")
formatted_sumstats = pd.concat(pd.read_csv(file, sep="\t", chunksize=100000), ignore_index=True) # usecols=selected_columns, 
formatted_sumstats

,SNP,A1,A2
0,rs568182971,A,G
1,rs112920234,T,G
2,rs569167217,A,C
3,rs536478188,T,G
4,rs61838556,C,A
...,...,...,...
27805933,rs564820004,T,C
27805934,rs530585350,G,T
27805935,rs556622038,TA,T
27805936,rs541687576,T,TA


In [15]:
import gzip
import pandas as pd

path = '/cluster/work/boeva/lrabuzin/deepcast/benchmarking_study/munged_sumstats.sumstats.gz'

file = gzip.open(path, "rt")
formatted_sumstats = pd.concat(pd.read_csv(file, sep="\t", chunksize=100000), ignore_index=True) # usecols=selected_columns, 
formatted_sumstats

,SNP,N,Z,A1,A2
0,rs568182971,NaN,NaN,NaN,NaN
1,rs112920234,259.0,0.458,T,G
2,rs569167217,NaN,NaN,NaN,NaN
3,rs536478188,NaN,NaN,NaN,NaN
4,rs61838556,NaN,NaN,NaN,NaN
...,...,...,...,...,...
27805933,rs564820004,NaN,NaN,NaN,NaN
27805934,rs530585350,NaN,NaN,NaN,NaN
27805935,rs556622038,NaN,NaN,NaN,NaN
27805936,rs541687576,NaN,NaN,NaN,NaN


In [16]:
print(formatted_sumstats.isna().sum())

SNP           7
N      15822907
Z      15822907
A1     15822907
A2     15822907
dtype: int64


In [17]:
27805938 - 15822907

11983031